## Imports Required Libraries

In [1]:
# ============================================
# 1. IMPORT LIBRARIES
# ============================================

import numpy as np
import pandas as pd

import random
import math
import time
import os

import matplotlib.pyplot as plt

import pulp

## Generate Synthetic Dataset

In [2]:
# ============================================
# 2. SYNTHETIC DATASET GENERATION
# ============================================

np.random.seed(42)

n_customers = 20

customers = pd.DataFrame({
    "customer_id": range(1, n_customers + 1),
    "x": np.random.uniform(0, 100, n_customers),
    "y": np.random.uniform(0, 100, n_customers),
    "demand": np.random.randint(1, 10, n_customers)
})

# Depot
depot = {
    "customer_id": 0,
    "x": 50,
    "y": 50,
    "demand": 0
}

print("Synthetic Dataset")
print("=================")
print(customers)

print("\nDepot:")
print(depot)

Synthetic Dataset
    customer_id          x          y  demand
0             1  37.454012  61.185289       7
1             2  95.071431  13.949386       8
2             3  73.199394  29.214465       3
3             4  59.865848  36.636184       1
4             5  15.601864  45.606998       4
5             6  15.599452  78.517596       2
6             7   5.808361  19.967378       8
7             8  86.617615  51.423444       4
8             9  60.111501  59.241457       2
9            10  70.807258   4.645041       6
10           11   2.058449  60.754485       6
11           12  96.990985  17.052412       4
12           13  83.244264   6.505159       6
13           14  21.233911  94.888554       2
14           15  18.182497  96.563203       2
15           16  18.340451  80.839735       4
16           17  30.424224  30.461377       8
17           18  52.475643   9.767211       7
18           19  43.194502  68.423303       9
19           20  29.122914  44.015249       8

Depot:
{'custom

## Dataset Generation Function

In [3]:
# ============================================
# 3. DATASET GENERATION FUNCTION
# ============================================

def generate_dataset(n_customers, seed=42):

    np.random.seed(seed)

    data = pd.DataFrame({
        "customer_id": range(1, n_customers + 1),
        "x": np.random.uniform(0, 100, n_customers),
        "y": np.random.uniform(0, 100, n_customers),
        "demand": np.random.randint(1, 10, n_customers)
    })

    return data


# Generate default dataset
customers = generate_dataset(20, seed=42)

customers.head()

,customer_id,x,y,demand
0,1,37.454012,61.185289,7
1,2,95.071431,13.949386,8
2,3,73.199394,29.214465,3
3,4,59.865848,36.636184,1
4,5,15.601864,45.606998,4


## Vehicle Configuration

In [4]:
# ============================================
# 4. VEHICLE CONFIGURATION
# ============================================

NUM_VEHICLES = 4
VEHICLE_CAPACITY = 30

print("Number of vehicles:", NUM_VEHICLES)
print("Vehicle capacity:", VEHICLE_CAPACITY)

Number of vehicles: 4
Vehicle capacity: 30


## Feasibility Check

In [5]:
# ============================================
# 5. FEASIBILITY CHECK
# ============================================

total_demand = customers["demand"].sum()
total_capacity = NUM_VEHICLES * VEHICLE_CAPACITY

print("Total demand:", total_demand)
print("Total vehicle capacity:", total_capacity)

if total_demand <= total_capacity:
    print("Problem is feasible.")
else:
    print("Problem is infeasible.")

Total demand: 101
Total vehicle capacity: 120
Problem is feasible.


## Distance Matrix

In [6]:
# ============================================
# 6. DISTANCE MATRIX
# ============================================

def calculate_distance_matrix(
    customers,
    depot_x=50,
    depot_y=50,
    rounded=False
):
    
    locations = [
        (depot_x, depot_y)
    ] + list(
        zip(customers["x"], customers["y"])
    )

    n = len(locations)

    distance_matrix = np.zeros((n, n))

    for i in range(n):
        for j in range(n):

            x1, y1 = locations[i]
            x2, y2 = locations[j]

            distance = np.sqrt(
                (x1 - x2) ** 2 +
                (y1 - y2) ** 2
            )

            if rounded:
                distance = int(distance + 0.5)

            distance_matrix[i][j] = distance

    return distance_matrix


distance_matrix = calculate_distance_matrix(
    customers
)

print("Distance matrix shape:", distance_matrix.shape)

Distance matrix shape: (21, 21)


## ILP Solver

In [7]:
# ============================================
# 7. ILP SOLVER
# ============================================

def solve_ilp(
    customers,
    distance_matrix,
    num_vehicles,
    vehicle_capacity,
    time_limit=60
):

    n = len(customers)

    nodes = range(n + 1)
    customer_nodes = range(1, n + 1)

    demands = {
        i: int(customers.iloc[i - 1]["demand"])
        for i in customer_nodes
    }

    # Create model
    model = pulp.LpProblem(
        "CVRP",
        pulp.LpMinimize
    )

    # ----------------------------------------
    # Binary decision variables
    # x[i,j] = 1 if vehicle travels i -> j
    # ----------------------------------------

    x = pulp.LpVariable.dicts(
        "x",
        [
            (i, j)
            for i in nodes
            for j in nodes
            if i != j
        ],
        cat="Binary"
    )

    # ----------------------------------------
    # Load variables
    # ----------------------------------------

    u = pulp.LpVariable.dicts(
        "u",
        customer_nodes,
        lowBound=0,
        upBound=vehicle_capacity,
        cat="Continuous"
    )

    # ----------------------------------------
    # Objective
    # Minimize total distance
    # ----------------------------------------

    model += pulp.lpSum(
        distance_matrix[i][j] * x[i, j]
        for i in nodes
        for j in nodes
        if i != j
    )

    # ----------------------------------------
    # Every customer must be visited exactly once
    # Incoming edge
    # ----------------------------------------

    for j in customer_nodes:

        model += pulp.lpSum(
            x[i, j]
            for i in nodes
            if i != j
        ) == 1

    # ----------------------------------------
    # Every customer must leave exactly once
    # ----------------------------------------

    for i in customer_nodes:

        model += pulp.lpSum(
            x[i, j]
            for j in nodes
            if i != j
        ) == 1

    # ----------------------------------------
    # Maximum number of vehicles
    # ----------------------------------------

    model += pulp.lpSum(
        x[0, j]
        for j in customer_nodes
    ) <= num_vehicles

    # ----------------------------------------
    # Number of vehicles leaving depot
    # must equal number returning
    # ----------------------------------------

    model += (
        pulp.lpSum(
            x[0, j]
            for j in customer_nodes
        )
        ==
        pulp.lpSum(
            x[i, 0]
            for i in customer_nodes
        )
    )

    # ----------------------------------------
    # Capacity constraints
    # ----------------------------------------

    for i in customer_nodes:

        model += u[i] >= demands[i]

        model += u[i] <= vehicle_capacity

    # ----------------------------------------
    # MTZ capacity constraints
    # ----------------------------------------

    for i in customer_nodes:

        for j in customer_nodes:

            if i != j:

                model += (
                    u[j]
                    >=
                    u[i]
                    + demands[j]
                    -
                    vehicle_capacity *
                    (1 - x[i, j])
                )

    # ----------------------------------------
    # Solve
    # ----------------------------------------

    solver = pulp.PULP_CBC_CMD(
        msg=False,
        timeLimit=time_limit
    )

    start_time = time.perf_counter()

    model.solve(solver)

    runtime = time.perf_counter() - start_time

    return model, x, u, runtime

## Run ILP

In [8]:
# ============================================
# 8. RUN ILP
# ============================================

model, x, u, ilp_runtime = solve_ilp(
    customers=customers,
    distance_matrix=distance_matrix,
    num_vehicles=NUM_VEHICLES,
    vehicle_capacity=VEHICLE_CAPACITY,
    time_limit=60
)

print("ILP Status:", pulp.LpStatus[model.status])

if model.status in [
    pulp.LpStatusOptimal,
    pulp.LpStatusNotSolved,
    pulp.LpStatusInfeasible,
    pulp.LpStatusUndefined,
    pulp.LpStatusInfeasible
]:
    print("ILP Objective:", pulp.value(model.objective))

print("ILP Runtime:", round(ilp_runtime, 4), "seconds")

ILP Status: Optimal
ILP Objective: 608.5694436641983
ILP Runtime: 60.0156 seconds


## ILP Route Extraction

In [9]:
# ============================================
# 9. EXTRACT ILP ROUTES
# ============================================

def extract_routes(model, x, n):

    routes = []

    for vehicle_start in range(1, n + 1):

        start_value = pulp.value(
            x[0, vehicle_start]
        )

        if start_value is not None and start_value > 0.5:

            route = [0, vehicle_start]

            current = vehicle_start

            while True:

                next_node = None

                for j in range(n + 1):

                    if j != current:

                        value = pulp.value(
                            x[current, j]
                        )

                        if value is not None and value > 0.5:

                            next_node = j
                            break

                if next_node is None:

                    raise ValueError(
                        f"Could not find next node from {current}"
                    )

                route.append(next_node)

                if next_node == 0:
                    break

                current = next_node

            routes.append(route)

    return routes

## Display ILP Routes

In [10]:
# ============================================
# 10. DISPLAY ILP ROUTES
# ============================================

ilp_routes = extract_routes(
    model,
    x,
    len(customers)
)

print("ILP Routes")
print("==========")

for i, route in enumerate(ilp_routes, start=1):

    print(
        f"Vehicle {i}: {route}"
    )

ILP Routes
Vehicle 1: [0, 1, 19, 4, 3, 0]
Vehicle 2: [0, 9, 16, 14, 15, 6, 11, 5, 20, 0]
Vehicle 3: [0, 13, 2, 12, 8, 0]
Vehicle 4: [0, 18, 10, 7, 17, 0]


## Route Distance and Load Functions

In [11]:
# ============================================
# 11. ROUTE CALCULATION FUNCTIONS
# ============================================

def calculate_route_distance(
    route,
    distance_matrix
):

    distance = 0

    for i in range(len(route) - 1):

        distance += distance_matrix[
            route[i]
        ][
            route[i + 1]
        ]

    return distance


def calculate_route_load(
    route,
    customers
):

    load = 0

    for node in route:

        if node != 0:

            load += int(
                customers.iloc[node - 1]["demand"]
            )

    return load


def calculate_total_distance(
    routes,
    distance_matrix
):

    return sum(
        calculate_route_distance(
            route,
            distance_matrix
        )
        for route in routes
    )

## Solution Validation

In [12]:
# ============================================
# 12. SOLUTION VALIDATION
# ============================================

def validate_solution(
    routes,
    customers,
    num_vehicles,
    vehicle_capacity
):

    errors = []

    n_customers = len(customers)

    # ----------------------------------------
    # Vehicle count
    # ----------------------------------------

    if len(routes) > num_vehicles:

        errors.append(
            f"Uses {len(routes)} vehicles, "
            f"maximum allowed is {num_vehicles}"
        )

    # ----------------------------------------
    # Check routes
    # ----------------------------------------

    visited_customers = []

    for vehicle_number, route in enumerate(
        routes,
        start=1
    ):

        # Start/end depot
        if route[0] != 0:

            errors.append(
                f"Vehicle {vehicle_number} "
                f"does not start at depot."
            )

        if route[-1] != 0:

            errors.append(
                f"Vehicle {vehicle_number} "
                f"does not end at depot."
            )

        # Check capacity
        load = calculate_route_load(
            route,
            customers
        )

        if load > vehicle_capacity:

            errors.append(
                f"Vehicle {vehicle_number} "
                f"exceeds capacity: {load}"
            )

        # Collect customers
        for node in route:

            if node != 0:

                visited_customers.append(node)

    # ----------------------------------------
    # Check all customers visited once
    # ----------------------------------------

    expected = set(
        range(1, n_customers + 1)
    )

    visited = set(
        visited_customers
    )

    if visited != expected:

        missing = expected - visited
        extra = visited - expected

        if missing:

            errors.append(
                f"Missing customers: {sorted(missing)}"
            )

        if extra:

            errors.append(
                f"Invalid customers: {sorted(extra)}"
            )

    # Duplicate customers
    if len(visited_customers) != len(
        set(visited_customers)
    ):

        errors.append(
            "Some customers are visited more than once."
        )

    # ----------------------------------------
    # Result
    # ----------------------------------------

    if len(errors) == 0:

        return True, [
            "All customers are visited exactly once.",
            "All routes start and end at the depot.",
            "All vehicle capacities are respected.",
            "Vehicle count is within the allowed limit."
        ]

    return False, errors

## Validate ILP

In [13]:
# ============================================
# 13. VALIDATE ILP
# ============================================

ilp_valid, ilp_messages = validate_solution(
    ilp_routes,
    customers,
    NUM_VEHICLES,
    VEHICLE_CAPACITY
)

print("ILP solution valid:", ilp_valid)

for message in ilp_messages:

    print("-", message)

ILP solution valid: True
- All customers are visited exactly once.
- All routes start and end at the depot.
- All vehicle capacities are respected.
- Vehicle count is within the allowed limit.


## Print ILP Route Details

In [14]:
# ============================================
# 14. ILP ROUTE DETAILS
# ============================================

print("ILP Route Details")
print("=================")

for i, route in enumerate(
    ilp_routes,
    start=1
):

    distance = calculate_route_distance(
        route,
        distance_matrix
    )

    load = calculate_route_load(
        route,
        customers
    )

    print(
        f"Vehicle {i}: "
        f"Route={route}, "
        f"Load={load}, "
        f"Distance={distance:.2f}"
    )

print(
    "\nTotal ILP distance:",
    calculate_total_distance(
        ilp_routes,
        distance_matrix
    )
)

ILP Route Details
Vehicle 1: Route=[0, 1, 19, 4, 3, 0], Load=20, Distance=108.35
Vehicle 2: Route=[0, 9, 16, 14, 15, 6, 11, 5, 20, 0], Load=30, Distance=174.76
Vehicle 3: Route=[0, 13, 2, 12, 8, 0], Load=22, Distance=144.92
Vehicle 4: Route=[0, 18, 10, 7, 17, 0], Load=29, Distance=180.54

Total ILP distance: 608.5694436641985


## Genetic Algorithm